# atari_50m — DQN vs. Agent0 on four Atari games (50M frames)

One plot per (game, reward setting): each of the four components
(`dqn_atari`, `dqn_real_atari`, `agent0_atari`, `agent0_real_atari`) sweeps
`ENV_HYPERS.GAME` over `battle_zone`, `pong`, `breakout`, `ms_pacman` at a
single seed - 16 runs, paired up into 4 games x 2 reward settings (`atari`,
`real_atari`) = 8 plots. Each plot overlays DQN (`tab:blue`), Agent-0
(`tab:red`), a solid black **Dopamine Ref** curve, and a solid tab:purple
**Endpoint Ref** curve (see below) for that (game, reward setting) pair.
Plots are titled by environment, e.g. `Pong`; the `real_atari` setting
gets a `Real-` prefix, e.g. `Real-Pong`. (Atari is cluster-scale - this
notebook expects results produced elsewhere; see the experiment README.)

**Dopamine Ref** is the mean of Dopamine's published "DQN (Adam + MSE in
JAX)" baseline over its 5 seeds, from
[`baselines/atari/data`](https://github.com/google/dopamine/tree/master/baselines/atari/data)
in [google/dopamine](https://github.com/google/dopamine) (Castro et al.,
["Dopamine: A Research Framework for Deep Reinforcement
Learning"](https://arxiv.org/abs/1812.06110), 2018). Cached locally as
`dopamine_dqn_reference.json`, whose `_source` key records the exact
per-game URLs it was pulled from.

**Endpoint Ref** is the mean over 10 seeds of coresets' DDQN baseline with
a uniform 100k-size replay buffer (the `truefinal/medium` config in
`coresets/experiments/atari-50M-endpoint`), interpolated onto a shared
frame grid. Cached locally as `coresets_dqn_100k_reference.json`, whose
`_source` key records the source config and results database.

In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

_HERE = Path.cwd()
_EXP_DIR = _HERE if (_HERE / "config.py").exists() else Path("experiments/atari_50m")
sys.path.insert(0, str(_EXP_DIR))
sys.path.insert(0, str(_EXP_DIR.resolve().parents[1]))

from experiment import load_result, load_runs
from analysis.plotting import min_max_normalize, plot_mean_ci, seed_grids_for, style

from config import EXPERIMENT

FONT_SIZE = 20   # 2x matplotlib's default (10), passed to every text call below


def grid_for(component, n=500):
    """A shared [0, total_steps] timestep grid from any run's length."""
    df = load_runs(EXPERIMENT, component)
    total_steps = len(load_result(EXPERIMENT, component, df["run_id"][0])["reward"])
    return np.linspace(0, total_steps, n)


def env_title(game, is_real):
    """The plot title for one game: e.g. "Pong", or "Real-Pong" for real_atari."""
    title = game.replace("_", " ").title()
    return f"Real-{title}" if is_real else title


_DOPAMINE_REFERENCE = json.loads(
    (_EXP_DIR / "dopamine_dqn_reference.json").read_text()
)


def dopamine_reference_curve(game):
    """Dopamine's published DQN (Adam + MSE in JAX) curve for `game`.

    Mean raw return over its 5 published seeds, one point per training
    iteration (1M frames each); see dopamine_dqn_reference.json.
    """
    ref = _DOPAMINE_REFERENCE[game]
    return np.array(ref["frames"]), np.array(ref["mean_return"])


_CORESETS_REFERENCE = json.loads(
    (_EXP_DIR / "coresets_dqn_100k_reference.json").read_text()
)


def coresets_reference_curve(game):
    """coresets' DDQN, uniform 100k-buffer curve for `game`.

    Mean return over its 10 seeds (truefinal/medium config), interpolated
    onto a shared frame grid; see coresets_dqn_100k_reference.json.
    """
    ref = _CORESETS_REFERENCE[game]
    return np.array(ref["frames"]), np.array(ref["mean_return"])


In [ ]:
HERE = Path.cwd()
RESULTS = HERE / "results" if (HERE / "results").exists() else Path("experiments/atari_50m/results")

GAMES = ["battle_zone", "pong", "breakout", "ms_pacman"]

# (dqn component, agent0 component, is real_atari)
PAIRS = [
    ("dqn_atari", "agent0_atari", False),
    ("dqn_real_atari", "agent0_real_atari", True),
]

# All components run the same number of steps, so they share one grid.
GRID = grid_for(PAIRS[0][0])

# GRID is in env steps; scale to frames (steps x frameskip) for the x-axis.
FRAMESKIP = load_runs(EXPERIMENT, PAIRS[0][0])["ENV_HYPERS.FRAMESKIP"][0]
FRAME_GRID = GRID * FRAMESKIP
FRAME_TICKS = np.array([10, 20, 30, 40, 50]) * 1_000_000
FRAME_TICK_LABELS = [f"{t // 1_000_000}M" for t in FRAME_TICKS]


In [ ]:
figures = {}
for dqn_component, agent0_component, is_real in PAIRS:
    dqn_df = load_runs(EXPERIMENT, dqn_component)
    agent0_df = load_runs(EXPERIMENT, agent0_component)
    for game in GAMES:
        dqn_run = dqn_df.filter(dqn_df["ENV_HYPERS.GAME"] == game)["run_id"][0]
        agent0_run = agent0_df.filter(agent0_df["ENV_HYPERS.GAME"] == game)["run_id"][0]
        dqn_stack = seed_grids_for(EXPERIMENT, dqn_component, GRID, run_ids=[dqn_run])
        agent0_stack = seed_grids_for(
            EXPERIMENT, agent0_component, GRID, run_ids=[agent0_run]
        )

        fig, ax = plt.subplots(figsize=(9, 6))   # 2:3 height:width
        plot_mean_ci(ax, FRAME_GRID, dqn_stack, "DQN", "tab:blue")
        plot_mean_ci(ax, FRAME_GRID, agent0_stack, "Agent-0", "tab:red")
        ref_frames, ref_return = dopamine_reference_curve(game)
        in_range = ref_frames <= FRAME_GRID[-1]  # match our training window
        ax.plot(
            ref_frames[in_range], ref_return[in_range], color="black",
            lw=2.5, label="Dopamine Ref",
        )
        cs_frames, cs_return = coresets_reference_curve(game)
        cs_in_range = cs_frames <= FRAME_GRID[-1]  # match our training window
        ax.plot(
            cs_frames[cs_in_range], cs_return[cs_in_range], color="tab:purple",
            lw=2.5, label="Endpoint Ref",
        )
        ax.set_title(env_title(game, is_real), fontsize=FONT_SIZE)
        ax.legend(loc="lower right", frameon=False, fontsize=FONT_SIZE)
        style(ax, xlabel="Frames")   # score range differs per game, so let it auto-scale
        ax.set_ylabel(
            "Return", rotation=90, ha="center", va="center",
            labelpad=20, fontsize=FONT_SIZE,
        )
        ax.set_xlabel(ax.get_xlabel(), fontsize=FONT_SIZE)
        ax.set_xticks(FRAME_TICKS)
        ax.set_xticklabels(FRAME_TICK_LABELS)
        ax.tick_params(labelsize=FONT_SIZE)
        fig.tight_layout()
        figures[f"{dqn_component}_{game}"] = fig

plt.show()


## Grid: DQN vs. Agent0, per game

2 rows x 4 columns: top row is `atari`, bottom row is `real_atari`; each
column is a game, titled by its environment (e.g. `Pong` / `Real-Pong`).
Each panel overlays DQN (`tab:blue`) against Agent-0 (`tab:red`) for that
(reward setting, game) pair.

In [ ]:
grid_fig, grid_axes = plt.subplots(2, 4, figsize=(20, 8), sharex=True)
for row, (dqn_component, agent0_component, is_real) in enumerate(PAIRS):
    dqn_df = load_runs(EXPERIMENT, dqn_component)
    agent0_df = load_runs(EXPERIMENT, agent0_component)
    for col, game in enumerate(GAMES):
        ax = grid_axes[row, col]
        dqn_run = dqn_df.filter(dqn_df["ENV_HYPERS.GAME"] == game)["run_id"][0]
        agent0_run = agent0_df.filter(agent0_df["ENV_HYPERS.GAME"] == game)["run_id"][0]
        dqn_stack = seed_grids_for(EXPERIMENT, dqn_component, GRID, run_ids=[dqn_run])
        agent0_stack = seed_grids_for(
            EXPERIMENT, agent0_component, GRID, run_ids=[agent0_run]
        )
        plot_mean_ci(ax, FRAME_GRID, dqn_stack, "DQN", "tab:blue")
        plot_mean_ci(ax, FRAME_GRID, agent0_stack, "Agent-0", "tab:red")
        style(ax, xlabel="", ylabel="")
        ax.set_title(env_title(game, is_real), fontsize=FONT_SIZE)
        if row == 1:
            ax.set_xlabel("Frames", fontsize=FONT_SIZE)
        if col == 0:
            ax.set_ylabel(
                "Return", rotation=90, ha="center", va="center",
                labelpad=20, fontsize=FONT_SIZE,
            )
        if row == 0 and col == 0:
            ax.legend(loc="upper left", frameon=False, fontsize=FONT_SIZE)
        ax.set_xticks([0, FRAME_GRID[-1]])
        ax.set_xticklabels(["0", "50M"])
        ax.tick_params(labelsize=FONT_SIZE)

grid_fig.tight_layout()
plt.show()


## Grid: Atari vs. Real-Atari, per agent and game

2 rows x 4 columns: top row is DQN, bottom row is Agent-0; each column is
a game, titled by agent and game (e.g. `DQN-Pong`). Each panel overlays the
`atari` component (`tab:green`) against `real_atari` (`tab:purple`) for
that (agent, game) pair.

In [ ]:
ROWS2 = [
    ("dqn_atari", "dqn_real_atari", "DQN"),
    ("agent0_atari", "agent0_real_atari", "Agent-0"),
]

reward_grid_fig, reward_grid_axes = plt.subplots(2, 4, figsize=(20, 8), sharex=True)
for row, (atari_component, real_component, agent_label) in enumerate(ROWS2):
    atari_df = load_runs(EXPERIMENT, atari_component)
    real_df = load_runs(EXPERIMENT, real_component)
    for col, game in enumerate(GAMES):
        ax = reward_grid_axes[row, col]
        atari_run = atari_df.filter(atari_df["ENV_HYPERS.GAME"] == game)["run_id"][0]
        real_run = real_df.filter(real_df["ENV_HYPERS.GAME"] == game)["run_id"][0]
        atari_stack = seed_grids_for(EXPERIMENT, atari_component, GRID, run_ids=[atari_run])
        real_stack = seed_grids_for(EXPERIMENT, real_component, GRID, run_ids=[real_run])
        plot_mean_ci(ax, FRAME_GRID, atari_stack, "Atari", "tab:green")
        plot_mean_ci(ax, FRAME_GRID, real_stack, "Real-Atari", "tab:purple")
        style(ax, xlabel="", ylabel="")
        game_title = game.replace("_", " ").title()
        ax.set_title(f"{agent_label}-{game_title}", fontsize=FONT_SIZE)
        if row == 1:
            ax.set_xlabel("Frames", fontsize=FONT_SIZE)
        if col == 0:
            ax.set_ylabel(
                "Return", rotation=90, ha="center", va="center",
                labelpad=20, fontsize=FONT_SIZE,
            )
        if row == 0 and col == 0:
            ax.legend(loc="upper left", frameon=False, fontsize=FONT_SIZE)
        ax.set_xticks([0, FRAME_GRID[-1]])
        ax.set_xticklabels(["0", "50M"])
        ax.tick_params(labelsize=FONT_SIZE)

reward_grid_fig.tight_layout()
plt.show()


## Aggregate: normalized return across games

Each game's return curves are min-max normalized with bounds shared by all
four components, then pooled across the four games. Each band is a 95%
bootstrap CI over the pooled runs (one per game at a single seed).

In [ ]:
COMPONENTS = {
    "dqn_atari": ("DQN", "tab:blue"),
    "dqn_real_atari": ("DQN (real Atari)", "tab:cyan"),
    "agent0_atari": ("Agent-0", "tab:red"),
    "agent0_real_atari": ("Agent-0 (real Atari)", "tab:orange"),
}

normalized = {component: [] for component in COMPONENTS}
for game in GAMES:
    stacks = []
    for component in COMPONENTS:
        df = load_runs(EXPERIMENT, component)
        run_ids = df.filter(df["ENV_HYPERS.GAME"] == game)["run_id"].to_list()
        stacks.append(seed_grids_for(EXPERIMENT, component, GRID, run_ids=run_ids))
    for component, stack in zip(COMPONENTS, min_max_normalize(stacks), strict=True):
        normalized[component].append(stack)

aggregate_fig, ax = plt.subplots(figsize=(9, 6))   # 2:3 height:width
for component, (label, color) in COMPONENTS.items():
    plot_mean_ci(ax, FRAME_GRID, np.vstack(normalized[component]), label, color)
ax.set_title("All games", fontsize=FONT_SIZE)
ax.legend(loc="upper left", frameon=False, fontsize=FONT_SIZE)
style(ax, ylim=(0, 1), xlabel="Frames")
ax.set_ylabel(
    "Normalized\nreturn", rotation=90, ha="center", va="center",
    labelpad=20, fontsize=FONT_SIZE,
)
ax.set_xlabel(ax.get_xlabel(), fontsize=FONT_SIZE)
ax.set_xticks(FRAME_TICKS)
ax.set_xticklabels(FRAME_TICK_LABELS)
ax.tick_params(labelsize=FONT_SIZE)
aggregate_fig.tight_layout()
plt.show()


In [ ]:
PLOTS_DIR = RESULTS.parent / "plots"
PLOTS_DIR.mkdir(exist_ok=True)
for name, fig in figures.items():
    fig.savefig(PLOTS_DIR / f"{name}.pdf", bbox_inches="tight")
grid_fig.savefig(PLOTS_DIR / "grid_dqn_vs_agent0.pdf", bbox_inches="tight")
reward_grid_fig.savefig(PLOTS_DIR / "grid_atari_vs_real_atari.pdf", bbox_inches="tight")
aggregate_fig.savefig(PLOTS_DIR / "aggregate_normalized_return.pdf", bbox_inches="tight")
print(f"saved {len(figures) + 3} plot(s) to {PLOTS_DIR}")
